In [1]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm


# ============================================================
# PATHS
# ============================================================

DRIVE_ROOT = Path(
    "/Users/marcodalbis/Library/CloudStorage/"
    "GoogleDrive-marco.dalbis@gmail.com/"
    "Il mio Drive/"
    "basketball-thesis"
)

SEQUENCE_NAME = "v_00HRwkvvjtQ_c001"

FRAMES_DIR = (
    DRIVE_ROOT
    / "datasets"
    / "sportsmot"
    / "val"
    / SEQUENCE_NAME
    / "img1"
)


# ============================================================
# EXPERIMENT OUTPUT
# ============================================================

OUTPUT_DIR = (
    DRIVE_ROOT
    / "experiments"
    / "sam2_dynamic_refresh"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# BASIC CHECKS
# ============================================================

print("Drive root:", DRIVE_ROOT)
print("Frames dir:", FRAMES_DIR)
print("Frames exists:", FRAMES_DIR.exists())
print("Output dir:", OUTPUT_DIR)

frame_files = sorted(
    FRAMES_DIR.glob("*.jpg")
)

print("Number of frames:", len(frame_files))

if len(frame_files) > 0:
    print("First frame:", frame_files[0].name)
    print("Last frame:", frame_files[-1].name)

Drive root: /Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis
Frames dir: /Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/datasets/sportsmot/val/v_00HRwkvvjtQ_c001/img1
Frames exists: True
Output dir: /Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/experiments/sam2_dynamic_refresh
Number of frames: 1165
First frame: 000001.jpg
Last frame: 001162.jpg


In [2]:
VALID_FRAME_PATHS = sorted(
    [
        path
        for path in FRAMES_DIR.glob("*.jpg")
        if path.stem.isdigit()
        and 1 <= int(path.stem) <= 1162
    ],
    key=lambda p: int(p.stem)
)

print(
    "Valid numeric frames:",
    len(VALID_FRAME_PATHS)
)

print(
    "Valid first:",
    VALID_FRAME_PATHS[0].name
)

print(
    "Valid last:",
    VALID_FRAME_PATHS[-1].name
)

Valid numeric frames: 1162
Valid first: 000001.jpg
Valid last: 001162.jpg


In [3]:
# ============================================================
# EXPERIMENT CONFIGURATION
# ============================================================

TEST_START_FRAME = 1
TEST_END_FRAME = 200

REFRESH_INTERVAL = 5

DETECTION_THRESHOLD = 0.30

SAM2_MATCH_IOU_THRESHOLD = 0.05

BOUNDARY_IOU_THRESHOLD = 0.05
BOUNDARY_CENTER_THRESHOLD = 0.06


print(
    "Test frames:",
    TEST_START_FRAME,
    "->",
    TEST_END_FRAME
)

print(
    "Refresh every:",
    REFRESH_INTERVAL,
    "frames"
)

print(
    "Refresh time:",
    REFRESH_INTERVAL / 25,
    "seconds"
)

Test frames: 1 -> 200
Refresh every: 5 frames
Refresh time: 0.2 seconds


In [4]:
# ============================================================
# LOAD RF-DETR
# ============================================================

from rfdetr import RFDETRNano


CHECKPOINT_PATH = (
    DRIVE_ROOT
    / "checkpoints"
    / "rfdetr_nano_10epochs"
    / "checkpoint_best_total.pth"
)


print(
    "Checkpoint:",
    CHECKPOINT_PATH
)

print(
    "Exists:",
    CHECKPOINT_PATH.exists()
)


detector = RFDETRNano(
    pretrain_weights=str(
        CHECKPOINT_PATH
    )
)

print(
    "RF-DETR loaded"
)

/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/.venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/.venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
[2026-10-08 08:12:32] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-10-08 08:12:32] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


Checkpoint: /Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/checkpoints/rfdetr_nano_10epochs/checkpoint_best_total.pth
Exists: True


[2026-10-08 08:13:05] [WARNING] rf-detr - Checkpoint has 1 classes but model is configured for 90. Using checkpoint class count (1). Pass num_classes=1 to suppress this warning.


RF-DETR loaded


In [5]:
# ============================================================
# RF-DETR DETECTION HELPER
# ============================================================

def detect_players(
    frame,
    threshold=DETECTION_THRESHOLD
):

    detections = detector.predict(
        frame,
        threshold=threshold
    )

    boxes = np.asarray(
        detections.xyxy,
        dtype=np.float32
    )

    if len(boxes) == 0:

        return (
            np.empty(
                (0, 4),
                dtype=np.float32
            ),
            np.empty(
                (0,),
                dtype=np.float32
            )
        )

    confidences = np.asarray(
        detections.confidence,
        dtype=np.float32
    )

    return (
        boxes,
        confidences
    )

In [6]:
# ============================================================
# RF-DETR SANITY CHECK
# ============================================================

test_frame = cv2.imread(
    str(
        FRAMES_DIR
        / "000001.jpg"
    )
)

test_boxes, test_confidences = (
    detect_players(
        test_frame
    )
)

print(
    "Detections:",
    len(test_boxes)
)

print(
    "Boxes shape:",
    test_boxes.shape
)

print(
    "Confidences:",
    test_confidences
)

[2026-10-08 08:13:06] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. For full GPU throughput (e.g. ~8x on T4 via FP16 Tensor Cores), call model.inference(dtype=torch.float16).


Detections: 9
Boxes shape: (9, 4)
Confidences: [0.9406879  0.9129167  0.88395554 0.8577601  0.84258366 0.7491012
 0.60921323 0.5973963  0.50771993]


In [7]:
# ============================================================
# RF-DETR CACHE - FIRST 200 FRAMES
# ============================================================

rfdetr_cache = {}


for frame_number in tqdm(
    range(
        TEST_START_FRAME,
        TEST_END_FRAME + 1
    ),
    desc="RF-DETR detections"
):

    frame_path = (
        FRAMES_DIR
        / f"{frame_number:06d}.jpg"
    )

    frame = cv2.imread(
        str(frame_path)
    )

    if frame is None:
        continue

    boxes, confidences = (
        detect_players(
            frame
        )
    )

    rfdetr_cache[
        frame_number
    ] = {
        "boxes": boxes,
        "confidences": confidences
    }


print(
    "Cached frames:",
    len(rfdetr_cache)
)

RF-DETR detections:   0%|          | 0/200 [00:00<?, ?it/s]

Cached frames: 200


In [8]:
# ============================================================
# DETECTION STATISTICS
# ============================================================

detection_counts = np.array(
    [
        len(
            rfdetr_cache[
                frame_number
            ]["boxes"]
        )
        for frame_number
        in sorted(rfdetr_cache)
    ]
)


print(
    "Mean players/frame:",
    detection_counts.mean()
)

print(
    "Min:",
    detection_counts.min()
)

print(
    "Max:",
    detection_counts.max()
)

print(
    "First 30 counts:"
)

print(
    detection_counts[:30]
)

Mean players/frame: 9.365
Min: 7
Max: 12
First 30 counts:
[ 9 10  8 10 10  9  9 10  9  8  9  9  9 10 10 12 11 10 10 10 10 11 10 11
 10 12 11 11 11 11]


In [9]:
# ============================================================
# BBOX IOU
# ============================================================

def bbox_iou(
    box_a,
    box_b
):

    x1 = max(
        box_a[0],
        box_b[0]
    )

    y1 = max(
        box_a[1],
        box_b[1]
    )

    x2 = min(
        box_a[2],
        box_b[2]
    )

    y2 = min(
        box_a[3],
        box_b[3]
    )

    intersection = max(
        0,
        x2 - x1
    ) * max(
        0,
        y2 - y1
    )

    area_a = max(
        0,
        box_a[2] - box_a[0]
    ) * max(
        0,
        box_a[3] - box_a[1]
    )

    area_b = max(
        0,
        box_b[2] - box_b[0]
    ) * max(
        0,
        box_b[3] - box_b[1]
    )

    union = (
        area_a
        + area_b
        - intersection
    )

    if union <= 0:
        return 0.0

    return (
        intersection / union
    )

In [10]:
# ============================================================
# NORMALIZED CENTER DISTANCE
# ============================================================

def normalized_center_distance(
    box_a,
    box_b,
    frame_width=1280,
    frame_height=720
):

    ax = (
        box_a[0]
        + box_a[2]
    ) / 2.0

    ay = (
        box_a[1]
        + box_a[3]
    ) / 2.0

    bx = (
        box_b[0]
        + box_b[2]
    ) / 2.0

    by = (
        box_b[1]
        + box_b[3]
    ) / 2.0

    distance = np.sqrt(
        (ax - bx) ** 2
        +
        (ay - by) ** 2
    )

    diagonal = np.sqrt(
        frame_width ** 2
        +
        frame_height ** 2
    )

    return (
        distance
        /
        diagonal
    )

In [11]:
# ============================================================
# BOX MATCHING
# ============================================================

from scipy.optimize import (
    linear_sum_assignment
)


def match_boxes(
    previous_boxes,
    current_boxes,
    iou_threshold=0.05,
    center_threshold=0.06
):

    if (
        len(previous_boxes) == 0
        or
        len(current_boxes) == 0
    ):

        return []


    cost_matrix = np.zeros(
        (
            len(previous_boxes),
            len(current_boxes)
        ),
        dtype=np.float32
    )

    iou_matrix = np.zeros_like(
        cost_matrix
    )

    center_matrix = np.zeros_like(
        cost_matrix
    )


    for i, previous_box in enumerate(
        previous_boxes
    ):

        for j, current_box in enumerate(
            current_boxes
        ):

            iou = bbox_iou(
                previous_box,
                current_box
            )

            center_distance = (
                normalized_center_distance(
                    previous_box,
                    current_box
                )
            )

            iou_matrix[i, j] = iou

            center_matrix[
                i,
                j
            ] = center_distance

            cost_matrix[
                i,
                j
            ] = (
                0.70 * (1 - iou)
                +
                0.30 * center_distance
            )


    row_indices, col_indices = (
        linear_sum_assignment(
            cost_matrix
        )
    )


    matches = []


    for i, j in zip(
        row_indices,
        col_indices
    ):

        iou = float(
            iou_matrix[i, j]
        )

        center_distance = float(
            center_matrix[i, j]
        )

        if (
            iou >= iou_threshold
            or
            center_distance
            <= center_threshold
        ):

            matches.append(
                (
                    int(i),
                    int(j),
                    iou,
                    center_distance
                )
            )


    return matches

In [12]:
# ============================================================
# LOAD SAM2
# ============================================================

import os
import sys


os.environ[
    "PYTORCH_ENABLE_MPS_FALLBACK"
] = "1"


SAM2_ROOT = Path(
    "/Users/marcodalbis/Desktop/"
    "Tesi Magistrale/"
    "sam2-official"
)

SAM2_CHECKPOINT = (
    SAM2_ROOT
    / "checkpoints"
    / "sam2.1_hiera_small.pt"
)

SAM2_CONFIG = (
    "configs/sam2.1/"
    "sam2.1_hiera_s.yaml"
)


if str(SAM2_ROOT) not in sys.path:

    sys.path.insert(
        0,
        str(SAM2_ROOT)
    )


from sam2.build_sam import (
    build_sam2_video_predictor
)


device = (
    torch.device("mps")
    if torch.backends.mps.is_available()
    else torch.device("cpu")
)


print(
    "Device:",
    device
)

print(
    "Checkpoint exists:",
    SAM2_CHECKPOINT.exists()
)


sam2_predictor = (
    build_sam2_video_predictor(
        SAM2_CONFIG,
        str(SAM2_CHECKPOINT),
        device=device
    )
)


print(
    "SAM2 loaded"
)

Device: mps
Checkpoint exists: True
SAM2 loaded


In [13]:
# ============================================================
# SAM2 MASK -> BBOX
# ============================================================

def mask_to_box(
    mask
):

    mask = np.asarray(
        mask
    ).squeeze()

    ys, xs = np.where(
        mask
    )

    if len(xs) == 0:
        return None

    x1 = float(
        xs.min()
    )

    y1 = float(
        ys.min()
    )

    x2 = float(
        xs.max() + 1
    )

    y2 = float(
        ys.max() + 1
    )

    return np.array(
        [
            x1,
            y1,
            x2,
            y2
        ],
        dtype=np.float32
    )

In [14]:
# ============================================================
# PERSISTENT SAM2 CONFIG
# ============================================================

PERSISTENT_TEST_START = 1
PERSISTENT_TEST_END = 200

REFRESH_INTERVAL = 5

NEW_TRACK_CONFIRM_FRAMES = 3

MATCH_IOU_THRESHOLD = 0.05
MATCH_CENTER_THRESHOLD = 0.05


print(
    "Persistent SAM2 test:",
    PERSISTENT_TEST_START,
    "->",
    PERSISTENT_TEST_END
)

print(
    "Refresh interval:",
    REFRESH_INTERVAL
)

print(
    "New track confirmation:",
    NEW_TRACK_CONFIRM_FRAMES,
    "frames"
)

Persistent SAM2 test: 1 -> 200
Refresh interval: 5
New track confirmation: 3 frames


In [15]:
# ============================================================
# CREATE ONE PERSISTENT VIDEO DIRECTORY
# local real JPEG copies - no symlinks
# ============================================================

import shutil


PERSISTENT_VIDEO_DIR = Path(
    "/tmp/basketball_sam2_persistent_001_200"
)

if PERSISTENT_VIDEO_DIR.exists():
    shutil.rmtree(
        PERSISTENT_VIDEO_DIR
    )

PERSISTENT_VIDEO_DIR.mkdir(
    parents=True,
    exist_ok=True
)


failed_frames = []


for local_idx, frame_number in tqdm(
    enumerate(
        range(
            PERSISTENT_TEST_START,
            PERSISTENT_TEST_END + 1
        )
    ),
    total=(
        PERSISTENT_TEST_END
        - PERSISTENT_TEST_START
        + 1
    ),
    desc="Preparing local JPEG frames"
):

    source = (
        FRAMES_DIR
        / f"{frame_number:06d}.jpg"
    )

    destination = (
        PERSISTENT_VIDEO_DIR
        / f"{local_idx:06d}.jpg"
    )

    frame = cv2.imread(
        str(source)
    )

    if frame is None:
        failed_frames.append(
            frame_number
        )
        continue

    success = cv2.imwrite(
        str(destination),
        frame,
        [
            cv2.IMWRITE_JPEG_QUALITY,
            95
        ]
    )

    if not success:
        failed_frames.append(
            frame_number
        )


print(
    "Persistent sequence:",
    PERSISTENT_VIDEO_DIR
)

print(
    "Frames written:",
    len(
        list(
            PERSISTENT_VIDEO_DIR.glob(
                "*.jpg"
            )
        )
    )
)

print(
    "Failed original frames:",
    failed_frames
)

Preparing local JPEG frames:   0%|          | 0/200 [00:00<?, ?it/s]

Persistent sequence: /tmp/basketball_sam2_persistent_001_200
Frames written: 200
Failed original frames: []


In [16]:
from PIL import Image

bad_files = []

for path in sorted(
    PERSISTENT_VIDEO_DIR.glob("*.jpg")
):
    try:
        with Image.open(path) as img:
            img.verify()
    except Exception as e:
        bad_files.append(
            (path.name, str(e))
        )

print("Bad JPEGs:", bad_files)

Bad JPEGs: []


In [17]:
# ============================================================
# INITIALIZE ONE SAM2 STATE
# ============================================================

persistent_state = (
    sam2_predictor.init_state(
        video_path=str(
            PERSISTENT_VIDEO_DIR
        )
    )
)

print(
    "Persistent SAM2 state initialized"
)

frame loading (JPEG): 100%|██████████| 200/200 [00:06<00:00, 33.13it/s]


Persistent SAM2 state initialized


In [18]:
# ============================================================
# INITIAL OBJECTS FROM FRAME 1
# ============================================================

first_boxes = (
    rfdetr_cache[
        PERSISTENT_TEST_START
    ]["boxes"]
)

next_track_id = 0


for box in first_boxes:

    sam2_predictor.add_new_points_or_box(
        inference_state=persistent_state,
        frame_idx=0,
        obj_id=next_track_id,
        box=np.asarray(
            box,
            dtype=np.float32
        )
    )

    next_track_id += 1


print(
    "Initial SAM2 objects:",
    len(first_boxes)
)

print(
    "Next track ID:",
    next_track_id
)

/Users/marcodalbis/Desktop/Tesi Magistrale/sam2-official/sam2/sam2_video_predictor.py:786: UserWarning: cannot import name '_C' from 'sam2' (/Users/marcodalbis/Desktop/Tesi Magistrale/sam2-official/sam2/__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(


Initial SAM2 objects: 9
Next track ID: 9


In [19]:
# ============================================================
# CANDIDATE STORAGE FOR NEW PLAYERS
# ============================================================

pending_candidates = []

persistent_rows = []

print(
    "Pending candidate system initialized"
)

Pending candidate system initialized


In [20]:
# ============================================================
# CHECK SAM2 VIDEO API
# ============================================================

import inspect


print("propagate_in_video:")
print(
    inspect.signature(
        sam2_predictor.propagate_in_video
    )
)

print()

print("add_new_points_or_box:")
print(
    inspect.signature(
        sam2_predictor.add_new_points_or_box
    )
)

propagate_in_video:
(inference_state, start_frame_idx=None, max_frame_num_to_track=None, reverse=False)

add_new_points_or_box:
(inference_state, frame_idx, obj_id, points=None, labels=None, clear_old_points=True, normalize_coords=True, box=None)


In [21]:
# ============================================================
# PERSISTENT TRACKING RUNTIME CONFIG
# ============================================================

MAX_MISSED_REFRESHES = 10
PENDING_MAX_GAP = 2

# Reset per poter rieseguire questa parte del notebook
persistent_rows = []
pending_candidates = []

track_missed_refreshes = {
    int(track_id): 0
    for track_id in range(next_track_id)
}

print(
    "Existing SAM2 IDs:",
    sorted(track_missed_refreshes.keys())
)

print(
    "Next ID:",
    next_track_id
)

print(
    "Max missed refreshes:",
    MAX_MISSED_REFRESHES
)

print(
    "Pending max gap:",
    PENDING_MAX_GAP
)

Existing SAM2 IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Next ID: 9
Max missed refreshes: 10
Pending max gap: 2


In [22]:
# ============================================================
# PENDING NEW-PLAYER CANDIDATES
# ============================================================

def update_pending_candidates(
    frame_number,
    detector_boxes,
    detector_confidences,
    unmatched_detection_indices
):

    global pending_candidates

    # --------------------------------------------------------
    # Elimina candidati spariti da troppo tempo
    # --------------------------------------------------------

    pending_candidates = [
        candidate
        for candidate in pending_candidates
        if (
            frame_number
            - candidate["last_frame"]
            <= PENDING_MAX_GAP
        )
    ]


    if len(unmatched_detection_indices) == 0:
        return


    unmatched_boxes = np.array(
        [
            detector_boxes[index]
            for index
            in unmatched_detection_indices
        ],
        dtype=np.float32
    )


    # --------------------------------------------------------
    # Prova ad associare le detection unmatched
    # ai candidati già esistenti
    # --------------------------------------------------------

    matched_current = set()


    if len(pending_candidates) > 0:

        candidate_boxes = np.array(
            [
                candidate["box"]
                for candidate
                in pending_candidates
            ],
            dtype=np.float32
        )


        candidate_matches = match_boxes(
            candidate_boxes,
            unmatched_boxes,
            iou_threshold=0.10,
            center_threshold=0.035
        )


        for (
            candidate_index,
            unmatched_local_index,
            _,
            _
        ) in candidate_matches:

            detection_index = (
                unmatched_detection_indices[
                    unmatched_local_index
                ]
            )

            box = np.array(
                detector_boxes[
                    detection_index
                ],
                dtype=np.float32
            )

            confidence = float(
                detector_confidences[
                    detection_index
                ]
            )

            candidate = (
                pending_candidates[
                    candidate_index
                ]
            )


            # Richiediamo continuità temporale ragionevole
            if (
                frame_number
                - candidate["last_frame"]
                <= PENDING_MAX_GAP
            ):

                candidate["box"] = box

                candidate["hits"] += 1

                candidate[
                    "last_frame"
                ] = frame_number

                candidate[
                    "history"
                ].append(
                    (
                        frame_number,
                        box.copy(),
                        confidence
                    )
                )

                matched_current.add(
                    unmatched_local_index
                )


    # --------------------------------------------------------
    # Detection che non corrispondono nemmeno a un candidate
    # --------------------------------------------------------

    for unmatched_local_index, detection_index in enumerate(
        unmatched_detection_indices
    ):

        if (
            unmatched_local_index
            in matched_current
        ):
            continue

        box = np.array(
            detector_boxes[
                detection_index
            ],
            dtype=np.float32
        )

        confidence = float(
            detector_confidences[
                detection_index
            ]
        )

        pending_candidates.append(
            {
                "box": box,
                "hits": 1,
                "first_frame": frame_number,
                "last_frame": frame_number,
                "history": [
                    (
                        frame_number,
                        box.copy(),
                        confidence
                    )
                ]
            }
        )

In [23]:
# ============================================================
# OUTPUT ROW HELPER
# ============================================================

def add_tracking_row(
    frame_number,
    track_id,
    box,
    confidence,
    source,
    sam2_iou=np.nan,
    center_distance=np.nan
):

    persistent_rows.append(
        {
            "frame":
                int(frame_number),

            "track_id":
                int(track_id),

            "x1":
                float(box[0]),

            "y1":
                float(box[1]),

            "x2":
                float(box[2]),

            "y2":
                float(box[3]),

            "confidence":
                float(confidence),

            "class_id":
                0,

            "source":
                source,

            "sam2_iou":
                sam2_iou,

            "center_distance":
                center_distance
        }
    )

In [24]:
# ============================================================
# PERSISTENT SAM2 + RF-DETR + LOST-TRACK REID RECOVERY
# ============================================================

# ============================================================
# 0. RESET PERSISTENT SAM2 STATE
# ============================================================

persistent_state = (
    sam2_predictor.init_state(
        video_path=str(
            PERSISTENT_VIDEO_DIR
        )
    )
)


# ============================================================
# 1. INITIAL OBJECTS FROM FIRST FRAME
# ============================================================

first_boxes = (
    rfdetr_cache[
        PERSISTENT_TEST_START
    ]["boxes"]
)

next_track_id = 0


for box in first_boxes:

    sam2_predictor.add_new_points_or_box(
        inference_state=
            persistent_state,

        frame_idx=0,

        obj_id=
            next_track_id,

        box=np.asarray(
            box,
            dtype=np.float32
        )
    )

    next_track_id += 1


print(
    "Initial SAM2 objects:",
    len(first_boxes)
)

print(
    "Next track ID:",
    next_track_id
)


# ============================================================
# 2. RESET OUTPUT / MEMORIES
# ============================================================

persistent_rows = []

pending_candidates = []

reid_gallery = {}

track_memory = {}


# ============================================================
# 3. INITIALIZE REID GALLERY + TRACK MEMORY
# ============================================================

for track_id, box in enumerate(
    first_boxes
):

    embedding = (
        extract_reid_embedding(
            frame_number=
                PERSISTENT_TEST_START,

            box=
                box
        )
    )

    add_embedding_to_gallery(
        reid_gallery,
        track_id,
        embedding
    )

    track_memory[
        int(track_id)
    ] = {

        "last_frame":
            int(
                PERSISTENT_TEST_START
            ),

        "last_box":
            np.asarray(
                box,
                dtype=np.float32
            ).copy(),

        "status":
            "active",

        "left_at_edge":
            is_near_frame_edge(
                box
            )
    }


track_missed_refreshes = {

    int(track_id): 0

    for track_id
    in track_memory.keys()
}


print(
    "Initial memory IDs:",
    sorted(
        track_memory.keys()
    )
)


# ============================================================
# 4. TEST RANGE
# ============================================================

NUM_TEST_FRAMES = (
    PERSISTENT_TEST_END
    - PERSISTENT_TEST_START
    + 1
)


segment_starts = range(
    0,
    NUM_TEST_FRAMES,
    REFRESH_INTERVAL
)


# ============================================================
# 5. MAIN LOOP
# ============================================================

for segment_start_idx in tqdm(
    segment_starts,
    desc="Persistent SAM2 + ReID tracking"
):

    # --------------------------------------------------------
    # Segment limits
    # --------------------------------------------------------

    segment_end_idx = min(
        segment_start_idx
        + REFRESH_INTERVAL
        - 1,
        NUM_TEST_FRAMES - 1
    )

    real_segment_start = (
        PERSISTENT_TEST_START
        + segment_start_idx
    )

    real_segment_end = (
        PERSISTENT_TEST_START
        + segment_end_idx
    )


    end_frame_matches = {}


    # ========================================================
    # SAM2 PROPAGATION
    # ========================================================

    max_frames_to_track = (
        segment_end_idx
        - segment_start_idx
    )


    for (
        local_frame_idx,
        object_ids,
        mask_logits
    ) in sam2_predictor.propagate_in_video(

        inference_state=
            persistent_state,

        start_frame_idx=
            segment_start_idx,

        max_frame_num_to_track=
            max_frames_to_track,

        reverse=False
    ):

        local_frame_idx = int(
            local_frame_idx
        )

        real_frame_number = (
            PERSISTENT_TEST_START
            + local_frame_idx
        )


        # ====================================================
        # RF-DETR DETECTIONS
        # ====================================================

        detections = (
            rfdetr_cache[
                real_frame_number
            ]
        )

        detector_boxes = (
            detections[
                "boxes"
            ]
        )

        detector_confidences = (
            detections[
                "confidences"
            ]
        )


        # ====================================================
        # SAM2 MASKS -> TEMPORARY BBOX
        # ====================================================

        sam2_boxes = []

        sam2_ids = []


        for object_index, object_id in enumerate(
            object_ids
        ):

            mask = (
                mask_logits[
                    object_index
                ] > 0.0
            )

            mask = (
                mask
                .detach()
                .cpu()
                .numpy()
            )

            box = (
                mask_to_box(
                    mask
                )
            )

            if box is None:
                continue


            sam2_boxes.append(
                box
            )

            sam2_ids.append(
                int(
                    object_id
                )
            )


        sam2_boxes = np.asarray(
            sam2_boxes,
            dtype=np.float32
        ).reshape(
            -1,
            4
        )


        used_detection_indices = set()


        # ====================================================
        # SAM2 -> RF-DETR ONE-TO-ONE MATCH
        # ====================================================

        if (
            len(sam2_boxes) > 0
            and
            len(detector_boxes) > 0
        ):

            matches = (
                match_boxes(
                    sam2_boxes,
                    detector_boxes,
                    iou_threshold=0.05,
                    center_threshold=0.05
                )
            )


            for (
                sam2_index,
                detection_index,
                match_iou,
                center_distance
            ) in matches:


                track_id = int(
                    sam2_ids[
                        sam2_index
                    ]
                )


                detector_box = np.asarray(
                    detector_boxes[
                        detection_index
                    ],
                    dtype=np.float32
                )


                confidence = float(
                    detector_confidences[
                        detection_index
                    ]
                )


                used_detection_indices.add(
                    detection_index
                )


                # ============================================
                # FAST MEMORY UPDATE
                #
                # NO OSNET HERE.
                # OSNet will run only at refresh.
                # ============================================

                update_track_memory(
                    track_id=
                        track_id,

                    frame_number=
                        real_frame_number,

                    detector_box=
                        detector_box
                )


                # ============================================
                # OUTPUT
                #
                # ID = SAM2
                # BOX = RF-DETR
                # ============================================

                add_tracking_row(

                    frame_number=
                        real_frame_number,

                    track_id=
                        track_id,

                    box=
                        detector_box,

                    confidence=
                        confidence,

                    source=
                        "sam2+rfdetr",

                    sam2_iou=
                        float(
                            match_iou
                        ),

                    center_distance=
                        float(
                            center_distance
                        )
                )


                # ============================================
                # Save match at refresh frame
                # ============================================

                if (
                    local_frame_idx
                    == segment_end_idx
                ):

                    end_frame_matches[
                        track_id
                    ] = (
                        detector_box.copy(),
                        confidence
                    )


        # ====================================================
        # RF-DETR DETECTIONS NOT EXPLAINED BY SAM2
        # ====================================================

        unmatched_detection_indices = [

            index

            for index in range(
                len(
                    detector_boxes
                )
            )

            if index
            not in used_detection_indices
        ]


        # ====================================================
        # UPDATE PENDING CANDIDATES
        # ====================================================

        update_pending_candidates(

            frame_number=
                real_frame_number,

            detector_boxes=
                detector_boxes,

            detector_confidences=
                detector_confidences,

            unmatched_detection_indices=
                unmatched_detection_indices
        )


    # ========================================================
    # REFRESH EXISTING SAM2 OBJECTS
    #
    # + PERIODIC REID GALLERY UPDATE
    # ========================================================

    for track_id, (
        detector_box,
        confidence
    ) in end_frame_matches.items():


        # ----------------------------------------------------
        # RF-DETR corrects SAM2 position
        # ----------------------------------------------------

        sam2_predictor.add_new_points_or_box(

            inference_state=
                persistent_state,

            frame_idx=
                segment_end_idx,

            obj_id=
                int(
                    track_id
                ),

            box=np.asarray(
                detector_box,
                dtype=np.float32
            )
        )


        # ----------------------------------------------------
        # Reset missed counter
        # ----------------------------------------------------

        track_missed_refreshes[
            int(
                track_id
            )
        ] = 0


        # ----------------------------------------------------
        # OSNet ONLY ON REFRESH
        # ----------------------------------------------------

        embedding = (
            extract_reid_embedding(

                frame_number=
                    real_segment_end,

                box=
                    detector_box
            )
        )


        add_embedding_to_gallery(

            reid_gallery,

            int(
                track_id
            ),

            embedding
        )


        # ----------------------------------------------------
        # Synchronize track memory
        # ----------------------------------------------------

        track_memory[
            int(
                track_id
            )
        ] = {

            "last_frame":
                int(
                    real_segment_end
                ),

            "last_box":
                np.asarray(
                    detector_box,
                    dtype=np.float32
                ).copy(),

            "status":
                "active",

            "left_at_edge":
                is_near_frame_edge(
                    detector_box
                )
        }


    # ========================================================
    # ACTIVE -> LOST
    # ========================================================

    for track_id in list(
        track_memory.keys()
    ):

        if (
            track_id
            not in end_frame_matches
        ):

            if (
                real_segment_end
                >
                track_memory[
                    track_id
                ][
                    "last_frame"
                ]
            ):

                track_memory[
                    track_id
                ][
                    "status"
                ] = "lost"


    # ========================================================
    # MISSED REFRESH COUNTERS
    # ========================================================

    current_sam2_ids = [

        int(
            track_id
        )

        for track_id
        in persistent_state[
            "obj_ids"
        ]
    ]


    ids_to_remove = []


    for track_id in current_sam2_ids:

        if (
            track_id
            in end_frame_matches
        ):
            continue


        track_missed_refreshes[
            track_id
        ] = (
            track_missed_refreshes.get(
                track_id,
                0
            )
            + 1
        )


        if (
            track_missed_refreshes[
                track_id
            ]
            >= MAX_MISSED_REFRESHES
        ):

            ids_to_remove.append(
                track_id
            )


    # ========================================================
    # REMOVE STALE OBJECTS FROM ACTIVE SAM2
    #
    # IMPORTANT:
    # track_memory + ReID gallery are preserved.
    # ========================================================

    for track_id in ids_to_remove:

        try:

            sam2_predictor.remove_object(

                inference_state=
                    persistent_state,

                obj_id=
                    int(
                        track_id
                    ),

                strict=False,

                need_output=False
            )

        except Exception as e:

            print(
                f"Could not remove ID "
                f"{track_id}: "
                f"{e}"
            )


        track_missed_refreshes.pop(
            track_id,
            None
        )


        if (
            track_id
            in track_memory
        ):

            track_memory[
                track_id
            ][
                "status"
            ] = "lost"


    # ========================================================
    # CONFIRM PENDING CANDIDATES
    # ========================================================

    confirmed_candidates = []


    for candidate in pending_candidates:

        if (
            candidate[
                "hits"
            ]
            >= NEW_TRACK_CONFIRM_FRAMES

            and

            candidate[
                "last_frame"
            ]
            == real_segment_end
        ):

            confirmed_candidates.append(
                candidate
            )


    # ========================================================
    # LOST TRACK RECOVERY
    #
    # BEFORE CREATING A NEW ID
    # ========================================================

    for candidate in confirmed_candidates:


        (
            recovered_id,
            reid_similarity
        ) = find_lost_track_match(

            candidate,
            real_segment_end
        )


        # ----------------------------------------------------
        # CASE A:
        # previously known player recovered
        # ----------------------------------------------------

        if (
            recovered_id
            is not None
        ):

            track_id = int(
                recovered_id
            )


            print(
                f"RECOVERED ID "
                f"{track_id}"
                f" | frame "
                f"{real_segment_end}"
                f" | ReID "
                f"{reid_similarity:.3f}"
            )


        # ----------------------------------------------------
        # CASE B:
        # genuinely new player
        # ----------------------------------------------------

        else:

            track_id = int(
                next_track_id
            )

            next_track_id += 1


            print(
                f"NEW ID "
                f"{track_id}"
                f" | frame "
                f"{real_segment_end}"
            )


        # ====================================================
        # ADD / RE-PROMPT OBJECT INTO SAME SAM2 STATE
        # ====================================================

        sam2_predictor.add_new_points_or_box(

            inference_state=
                persistent_state,

            frame_idx=
                segment_end_idx,

            obj_id=
                int(
                    track_id
                ),

            box=np.asarray(
                candidate[
                    "box"
                ],
                dtype=np.float32
            )
        )


        # ====================================================
        # RESET MISSED COUNTER
        # ====================================================

        track_missed_refreshes[
            track_id
        ] = 0


        # ====================================================
        # UPDATE MEMORY
        # ====================================================

        track_memory[
            track_id
        ] = {

            "last_frame":
                int(
                    real_segment_end
                ),

            "last_box":
                np.asarray(
                    candidate[
                        "box"
                    ],
                    dtype=np.float32
                ).copy(),

            "status":
                "active",

            "left_at_edge":
                is_near_frame_edge(
                    candidate[
                        "box"
                    ]
                )
        }


        # ====================================================
        # UPDATE REID GALLERY
        # ====================================================

        candidate_embedding = (
            get_candidate_embedding(
                candidate
            )
        )


        add_embedding_to_gallery(

            reid_gallery,

            track_id,

            candidate_embedding
        )


        # ====================================================
        # BACKFILL CONFIRMATION FRAMES
        # ====================================================

        for (
            history_frame,
            history_box,
            history_confidence
        ) in candidate[
            "history"
        ]:

            add_tracking_row(

                frame_number=
                    history_frame,

                track_id=
                    track_id,

                box=
                    history_box,

                confidence=
                    history_confidence,

                source=(
                    "reid_recovered"

                    if recovered_id
                    is not None

                    else

                    "new_confirmed"
                )
            )


        pending_candidates.remove(
            candidate
        )


    # ========================================================
    # PROGRESS LOG
    # ========================================================

    active_count = sum(

        1

        for memory
        in track_memory.values()

        if memory[
            "status"
        ]
        == "active"
    )


    lost_count = sum(

        1

        for memory
        in track_memory.values()

        if memory[
            "status"
        ]
        == "lost"
    )


    print(
        f"\nFrames "
        f"{real_segment_start}-"
        f"{real_segment_end}"
        f" | SAM2 IDs: "
        f"{len(persistent_state['obj_ids'])}"
        f" | active: "
        f"{active_count}"
        f" | lost: "
        f"{lost_count}"
        f" | next ID: "
        f"{next_track_id}"
        f" | pending: "
        f"{len(pending_candidates)}"
    )


print()
print("=" * 60)
print("TRACKING FINISHED")
print("=" * 60)

print(
    "Rows collected:",
    len(
        persistent_rows
    )
)

print(
    "Total identities created:",
    next_track_id
)

frame loading (JPEG): 100%|██████████| 200/200 [00:05<00:00, 37.41it/s]


Initial SAM2 objects: 9
Next track ID: 9


NameError: name 'extract_reid_embedding' is not defined

In [44]:
# ============================================================
# BUILD FINAL DATAFRAME
# ============================================================

persistent_df = pd.DataFrame(
    persistent_rows
)


persistent_df = (
    persistent_df
    .drop_duplicates(
        subset=[
            "frame",
            "track_id"
        ],
        keep="last"
    )
    .sort_values(
        [
            "frame",
            "track_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "Rows:",
    len(persistent_df)
)

print(
    "Unique IDs:",
    persistent_df[
        "track_id"
    ].nunique()
)

print(
    "Frame range:",
    persistent_df[
        "frame"
    ].min(),
    "->",
    persistent_df[
        "frame"
    ].max()
)

print(
    "Duplicate frame/ID:",
    persistent_df.duplicated(
        [
            "frame",
            "track_id"
        ]
    ).sum()
)

Rows: 1855
Unique IDs: 14
Frame range: 1 -> 200
Duplicate frame/ID: 0


In [45]:
# ============================================================
# TRACK STATISTICS
# ============================================================

track_stats = (
    persistent_df
    .groupby(
        "track_id"
    )
    .agg(
        observations=(
            "frame",
            "size"
        ),

        start_frame=(
            "frame",
            "min"
        ),

        end_frame=(
            "frame",
            "max"
        )
    )
    .sort_values(
        "start_frame"
    )
)


display(
    track_stats
)


print()

print(
    "Total IDs:",
    len(track_stats)
)

,observations,start_frame,end_frame
track_id,,,
0,197,1,200
1,198,1,200
2,76,1,76
3,180,1,200
4,197,1,200
5,186,1,200
6,151,1,200
7,88,1,107
8,54,1,55



Total IDs: 14


In [46]:
# ============================================================
# SOURCE DIAGNOSTIC
# ============================================================

print(
    persistent_df[
        "source"
    ].value_counts()
)

print()

print(
    persistent_df[
        "source"
    ].value_counts(
        normalize=True
    )
)

source
sam2+rfdetr      1832
new_confirmed      23
Name: count, dtype: int64

source
sam2+rfdetr      0.987601
new_confirmed    0.012399
Name: proportion, dtype: float64


In [47]:
# ============================================================
# TRACKS PER FRAME
# ============================================================

tracks_per_frame = (
    persistent_df
    .groupby(
        "frame"
    )
    ["track_id"]
    .nunique()
)


print(
    "Mean:",
    tracks_per_frame.mean()
)

print(
    "Min:",
    tracks_per_frame.min()
)

print(
    "Max:",
    tracks_per_frame.max()
)

print()

display(
    tracks_per_frame.head(
        30
    )
)

Mean: 9.275
Min: 7
Max: 11



frame
1      9
2     10
3      8
4     10
5     10
6      9
7      9
8     10
9      9
10     8
11     9
12     9
13     9
14    10
15    10
16    11
17    11
18    10
19    10
20    10
21    10
22    11
23    10
24    11
25    10
26    11
27    11
28    11
29    11
30    11
Name: track_id, dtype: int64

In [48]:
# ============================================================
# SAVE PERSISTENT TEST
# ============================================================

PERSISTENT_CSV = (
    OUTPUT_DIR
    / (
        f"{SEQUENCE_NAME}_"
        f"persistent_refresh5_"
        f"frames_001_200.csv"
    )
)


persistent_df.to_csv(
    PERSISTENT_CSV,
    index=False
)


print(
    "Saved:",
    PERSISTENT_CSV
)

print(
    "Exists:",
    PERSISTENT_CSV.exists()
)

Saved: /Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/experiments/sam2_dynamic_refresh/v_00HRwkvvjtQ_c001_persistent_refresh5_frames_001_200.csv
Exists: True


In [49]:
# ============================================================
# CREATE PERSISTENT TRACKING VIDEO
# ============================================================

PERSISTENT_VIDEO_OUTPUT = (
    OUTPUT_DIR
    / (
        f"{SEQUENCE_NAME}_"
        f"persistent_refresh5_"
        f"frames_001_200.mp4"
    )
)


first_frame = cv2.imread(
    str(
        FRAMES_DIR
        / "000001.jpg"
    )
)

height, width = (
    first_frame.shape[:2]
)


writer = cv2.VideoWriter(
    str(
        PERSISTENT_VIDEO_OUTPUT
    ),
    cv2.VideoWriter_fourcc(
        *"mp4v"
    ),
    25,
    (
        width,
        height
    )
)


for frame_number in tqdm(
    range(
        PERSISTENT_TEST_START,
        PERSISTENT_TEST_END + 1
    ),
    desc="Creating persistent video"
):

    frame = cv2.imread(
        str(
            FRAMES_DIR
            / f"{frame_number:06d}.jpg"
        )
    )

    if frame is None:
        continue


    frame_rows = (
        persistent_df[
            persistent_df[
                "frame"
            ]
            == frame_number
        ]
    )


    for _, row in frame_rows.iterrows():

        x1 = int(row["x1"])
        y1 = int(row["y1"])
        x2 = int(row["x2"])
        y2 = int(row["y2"])

        track_id = int(
            row["track_id"]
        )


        cv2.rectangle(
            frame,
            (
                x1,
                y1
            ),
            (
                x2,
                y2
            ),
            (
                255,
                255,
                255
            ),
            2
        )


        cv2.putText(
            frame,
            f"ID {track_id}",
            (
                x1,
                max(
                    20,
                    y1 - 8
                )
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (
                255,
                255,
                255
            ),
            2,
            cv2.LINE_AA
        )


    cv2.putText(
        frame,
        f"Frame {frame_number}",
        (
            20,
            35
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (
            255,
            255,
            255
        ),
        2,
        cv2.LINE_AA
    )


    writer.write(
        frame
    )


writer.release()


print(
    "Video:",
    PERSISTENT_VIDEO_OUTPUT
)

print(
    "Exists:",
    PERSISTENT_VIDEO_OUTPUT.exists()
)

Creating persistent video:   0%|          | 0/200 [00:00<?, ?it/s]

Video: /Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/experiments/sam2_dynamic_refresh/v_00HRwkvvjtQ_c001_persistent_refresh5_frames_001_200.mp4
Exists: True


In [51]:
# ============================================================
# OPEN VIDEO
# ============================================================

import subprocess


subprocess.run(
    [
        "open",
        str(
            PERSISTENT_VIDEO_OUTPUT
        )
    ]
)

CompletedProcess(args=['open', '/Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/experiments/sam2_dynamic_refresh/v_00HRwkvvjtQ_c001_persistent_refresh5_frames_001_200.mp4'], returncode=0)

In [52]:
# ============================================================
# LOST TRACK + REID CONFIG
# ============================================================

SHORT_LOST_MAX_FRAMES = 25       # 1 secondo
LONG_LOST_MAX_FRAMES = 150       # 6 secondi

NEW_TRACK_CONFIRM_FRAMES = 3
PENDING_MAX_GAP = 2

# ReID
SHORT_REID_THRESHOLD = 0.72
LONG_REID_THRESHOLD = 0.78

# Per occlusioni brevi
SHORT_CENTER_THRESHOLD = 0.15

# Quando considero una persona vicina al bordo
EDGE_MARGIN = 80

# Gallery: non accumuliamo infiniti embedding
MAX_REID_GALLERY = 20


print("Short lost memory:", SHORT_LOST_MAX_FRAMES)
print("Long lost memory:", LONG_LOST_MAX_FRAMES)
print("Short ReID threshold:", SHORT_REID_THRESHOLD)
print("Long ReID threshold:", LONG_REID_THRESHOLD)

Short lost memory: 25
Long lost memory: 150
Short ReID threshold: 0.72
Long ReID threshold: 0.78


In [53]:
# ============================================================
# LOAD OSNET REID
# ============================================================

from boxmot.reid.backends.pytorch_backend import PyTorchBackend


OSNET_WEIGHTS = Path(
    "/Users/marcodalbis/Desktop/Tesi Magistrale/"
    "basketball-thesis/basketball-thesis/"
    ".venv/lib/python3.11/site-packages/"
    "models/osnet_x0_25_msmt17.pt"
)


reid_device = (
    torch.device("mps")
    if torch.backends.mps.is_available()
    else torch.device("cpu")
)


reid_backend = PyTorchBackend(
    weights=OSNET_WEIGHTS,
    device=reid_device,
    half=False
)


print("ReID device:", reid_device)
print("Weights:", OSNET_WEIGHTS)
print("Exists:", OSNET_WEIGHTS.exists())

INFO     /Users/marcodalbis/Desktop/Tesi                                                                           
         Magistrale/basketball-thesis/basketball-thesis/.venv/lib/python3.11/site-packages/models/osnet_x0_25_msmt1
         7.pt

INFO     [PID 4652] Found existing ReID weights at /Users/marcodalbis/Desktop/Tesi                                 
         Magistrale/basketball-thesis/basketball-thesis/.venv/lib/python3.11/site-packages/models/osnet_x0_25_msmt1
         7.pt; skipping download.

INFO     Loaded deployed weights from /Users/marcodalbis/Desktop/Tesi                                              
         Magistrale/basketball-thesis/basketball-thesis/.venv/lib/python3.11/site-packages/models/osnet_x0_25_msmt1
         7.pt: 565/565 required tensors, 100.00% required elements

ReID device: mps
Weights: /Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/.venv/lib/python3.11/site-packages/models/osnet_x0_25_msmt17.pt
Exists: True


In [39]:
# ============================================================
# REID EMBEDDING
# ============================================================

def extract_reid_embedding(
    frame_number,
    box
):

    frame_path = (
        FRAMES_DIR
        / f"{int(frame_number):06d}.jpg"
    )

    frame = cv2.imread(
        str(frame_path)
    )

    if frame is None:
        return None

    h, w = frame.shape[:2]

    x1 = max(
        0.0,
        float(box[0])
    )

    y1 = max(
        0.0,
        float(box[1])
    )

    x2 = min(
        float(w),
        float(box[2])
    )

    y2 = min(
        float(h),
        float(box[3])
    )

    if x2 <= x1 or y2 <= y1:
        return None

    boxes = np.array(
        [[x1, y1, x2, y2]],
        dtype=np.float32
    )

    features = reid_backend.get_features(
        boxes,
        frame
    )

    features = np.asarray(
        features,
        dtype=np.float32
    )

    if features.size == 0:
        return None

    embedding = (
        features[0]
        .reshape(-1)
    )

    norm = np.linalg.norm(
        embedding
    )

    if norm > 0:
        embedding = embedding / norm

    return embedding

In [44]:
# ============================================================
# REID GALLERY HELPERS
# ============================================================

def add_embedding_to_gallery(
    gallery,
    track_id,
    embedding
):

    if embedding is None:
        return

    track_id = int(track_id)

    if track_id not in gallery:
        gallery[track_id] = []

    gallery[track_id].append(
        embedding
    )

    gallery[track_id] = (
        gallery[track_id][
            -MAX_REID_GALLERY:
        ]
    )


def get_gallery_embedding(
    gallery,
    track_id
):

    track_id = int(track_id)

    embeddings = gallery.get(
        track_id,
        []
    )

    if len(embeddings) == 0:
        return None

    mean_embedding = np.mean(
        np.stack(
            embeddings,
            axis=0
        ),
        axis=0
    )

    norm = np.linalg.norm(
        mean_embedding
    )

    if norm > 0:
        mean_embedding = (
            mean_embedding / norm
        )

    return mean_embedding


def cosine_similarity(
    embedding_a,
    embedding_b
):

    if (
        embedding_a is None
        or embedding_b is None
    ):
        return None

    return float(
        np.dot(
            embedding_a,
            embedding_b
        )
    )

In [17]:
required_names = [
    "persistent_state",
    "first_boxes",
    "next_track_id",
    "extract_reid_embedding",
    "add_embedding_to_gallery",
    "get_gallery_embedding",
    "get_candidate_embedding",
    "cosine_similarity",
    "is_near_frame_edge",
    "find_lost_track_match",
    "update_track_memory",
    "update_pending_candidates",
    "add_tracking_row",
    "mask_to_box",
    "match_boxes",
]

missing = [
    name
    for name in required_names
    if name not in globals()
]

print("Missing:", missing)

Missing: ['persistent_state', 'first_boxes', 'next_track_id', 'extract_reid_embedding', 'add_embedding_to_gallery', 'get_gallery_embedding', 'get_candidate_embedding', 'cosine_similarity', 'is_near_frame_edge', 'find_lost_track_match', 'update_track_memory', 'update_pending_candidates', 'add_tracking_row', 'mask_to_box', 'match_boxes']


In [56]:
# ============================================================
# EDGE EXIT DETECTION
# ============================================================

def is_near_frame_edge(
    box,
    width=1280,
    height=720,
    margin=EDGE_MARGIN
):

    x1, y1, x2, y2 = box

    return (
        x1 <= margin
        or
        y1 <= margin
        or
        x2 >= width - margin
        or
        y2 >= height - margin
    )

In [57]:
# ============================================================
# TRACK MEMORY
# ============================================================

reid_gallery = {}

track_memory = {}

pending_candidates = []

persistent_rows = []


# I 9 ID iniziali
for track_id, box in enumerate(
    first_boxes
):

    embedding = extract_reid_embedding(
        frame_number=1,
        box=box
    )

    add_embedding_to_gallery(
        reid_gallery,
        track_id,
        embedding
    )

    track_memory[track_id] = {
        "last_frame": 1,
        "last_box": np.array(
            box,
            dtype=np.float32
        ),
        "status": "active",
        "left_at_edge": is_near_frame_edge(
            box
        )
    }


print(
    "Initial memory IDs:",
    sorted(track_memory)
)

Initial memory IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8]


In [58]:
# ============================================================
# UPDATE NEW PLAYER CANDIDATES
# ============================================================

def update_pending_candidates(
    frame_number,
    detector_boxes,
    detector_confidences,
    unmatched_detection_indices
):

    global pending_candidates


    pending_candidates = [
        candidate
        for candidate in pending_candidates
        if (
            frame_number
            - candidate["last_frame"]
            <= PENDING_MAX_GAP
        )
    ]


    if len(unmatched_detection_indices) == 0:
        return


    unmatched_boxes = np.array(
        [
            detector_boxes[index]
            for index
            in unmatched_detection_indices
        ],
        dtype=np.float32
    )


    matched_current = set()


    if len(pending_candidates) > 0:

        candidate_boxes = np.array(
            [
                candidate["box"]
                for candidate
                in pending_candidates
            ],
            dtype=np.float32
        )

        matches = match_boxes(
            candidate_boxes,
            unmatched_boxes,
            iou_threshold=0.05,
            center_threshold=0.05
        )


        for (
            candidate_index,
            unmatched_local_index,
            _,
            _
        ) in matches:

            detection_index = (
                unmatched_detection_indices[
                    unmatched_local_index
                ]
            )

            candidate = (
                pending_candidates[
                    candidate_index
                ]
            )

            box = np.array(
                detector_boxes[
                    detection_index
                ],
                dtype=np.float32
            )

            confidence = float(
                detector_confidences[
                    detection_index
                ]
            )


            embedding = (
                extract_reid_embedding(
                    frame_number,
                    box
                )
            )


            candidate["box"] = box

            candidate["hits"] += 1

            candidate["last_frame"] = (
                frame_number
            )

            candidate[
                "history"
            ].append(
                (
                    frame_number,
                    box.copy(),
                    confidence
                )
            )

            if embedding is not None:
                candidate[
                    "embeddings"
                ].append(
                    embedding
                )

            matched_current.add(
                unmatched_local_index
            )


    for unmatched_local_index, detection_index in enumerate(
        unmatched_detection_indices
    ):

        if (
            unmatched_local_index
            in matched_current
        ):
            continue

        box = np.array(
            detector_boxes[
                detection_index
            ],
            dtype=np.float32
        )

        confidence = float(
            detector_confidences[
                detection_index
            ]
        )

        embedding = extract_reid_embedding(
            frame_number,
            box
        )


        pending_candidates.append(
            {
                "box": box,

                "hits": 1,

                "first_frame":
                    frame_number,

                "last_frame":
                    frame_number,

                "history": [
                    (
                        frame_number,
                        box.copy(),
                        confidence
                    )
                ],

                "embeddings": (
                    []
                    if embedding is None
                    else [embedding]
                )
            }
        )

In [59]:
# ============================================================
# CANDIDATE REID EMBEDDING
# ============================================================

def get_candidate_embedding(
    candidate
):

    embeddings = candidate[
        "embeddings"
    ]

    if len(embeddings) == 0:
        return None

    embedding = np.mean(
        np.stack(
            embeddings,
            axis=0
        ),
        axis=0
    )

    norm = np.linalg.norm(
        embedding
    )

    if norm > 0:
        embedding = (
            embedding / norm
        )

    return embedding

In [60]:
# ============================================================
# LOST TRACK RECOVERY
# ============================================================

def find_lost_track_match(
    candidate,
    current_frame
):

    candidate_embedding = (
        get_candidate_embedding(
            candidate
        )
    )

    if candidate_embedding is None:
        return None, None


    candidate_box = (
        candidate["box"]
    )


    best_track_id = None
    best_similarity = -1.0


    for track_id, memory in (
        track_memory.items()
    ):

        if memory[
            "status"
        ] == "active":
            continue


        gap = (
            current_frame
            - memory["last_frame"]
        )


        # --------------------------------------------
        # Quanto a lungo manteniamo questa identità?
        # --------------------------------------------

        if memory["left_at_edge"]:

            max_gap = (
                LONG_LOST_MAX_FRAMES
            )

            reid_threshold = (
                LONG_REID_THRESHOLD
            )

        else:

            max_gap = (
                SHORT_LOST_MAX_FRAMES
            )

            reid_threshold = (
                SHORT_REID_THRESHOLD
            )


        if gap > max_gap:
            continue


        old_embedding = (
            get_gallery_embedding(
                reid_gallery,
                track_id
            )
        )


        similarity = (
            cosine_similarity(
                candidate_embedding,
                old_embedding
            )
        )


        if similarity is None:
            continue


        # --------------------------------------------
        # OCCLUSIONE BREVE:
        # ReID + posizione devono essere plausibili
        # --------------------------------------------

        if not memory[
            "left_at_edge"
        ]:

            center_distance = (
                normalized_center_distance(
                    memory[
                        "last_box"
                    ],
                    candidate_box
                )
            )

            if (
                center_distance
                >
                SHORT_CENTER_THRESHOLD
            ):
                continue


        # --------------------------------------------
        # USCITA DAL BORDO:
        # posizione NON è vincolante
        # --------------------------------------------

        if (
            similarity
            >= reid_threshold

            and

            similarity
            > best_similarity
        ):

            best_similarity = (
                similarity
            )

            best_track_id = (
                int(track_id)
            )


    return (
        best_track_id,
        best_similarity
    )

In [37]:
# ============================================================
# UPDATE MEMORY FOR MATCHED TRACKS
# NO REID INFERENCE EVERY FRAME
# ============================================================

def update_track_memory(
    track_id,
    frame_number,
    detector_box
):

    track_memory[
        int(track_id)
    ] = {
        "last_frame":
            int(frame_number),

        "last_box":
            np.asarray(
                detector_box,
                dtype=np.float32
            ).copy(),

        "status":
            "active",

        "left_at_edge":
            is_near_frame_edge(
                detector_box
            )
    }

In [42]:
# ============================================================
# LOAD OSNET REID BACKEND - COMPATIBLE WITH CURRENT BOXMOT
# ============================================================

from pathlib import Path
import torch

from boxmot.reid.backends.pytorch_backend import PyTorchBackend


OSNET_WEIGHTS = Path(
    "/Users/marcodalbis/Desktop/Tesi Magistrale/"
    "basketball-thesis/basketball-thesis/"
    ".venv/lib/python3.11/site-packages/"
    "models/osnet_x0_25_msmt17.pt"
)


REID_DEVICE = (
    torch.device("mps")
    if torch.backends.mps.is_available()
    else torch.device("cpu")
)


print("ReID device:", REID_DEVICE)
print("Weights:", OSNET_WEIGHTS)
print("Weights exist:", OSNET_WEIGHTS.exists())


reid_backend = PyTorchBackend(
    weights=OSNET_WEIGHTS,
    device=REID_DEVICE,
    half=False
)


print()
print("ReID backend loaded:")
print(type(reid_backend))

ReID device: mps
Weights: /Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/.venv/lib/python3.11/site-packages/models/osnet_x0_25_msmt17.pt
Weights exist: True


INFO     /Users/marcodalbis/Desktop/Tesi                                                                           
         Magistrale/basketball-thesis/basketball-thesis/.venv/lib/python3.11/site-packages/models/osnet_x0_25_msmt1
         7.pt

INFO     [PID 11127] Found existing ReID weights at /Users/marcodalbis/Desktop/Tesi                                
         Magistrale/basketball-thesis/basketball-thesis/.venv/lib/python3.11/site-packages/models/osnet_x0_25_msmt1
         7.pt; skipping download.

INFO     Loaded deployed weights from /Users/marcodalbis/Desktop/Tesi                                              
         Magistrale/basketball-thesis/basketball-thesis/.venv/lib/python3.11/site-packages/models/osnet_x0_25_msmt1
         7.pt: 565/565 required tensors, 100.00% required elements


ReID backend loaded:
<class 'boxmot.reid.backends.pytorch_backend.PyTorchBackend'>


In [43]:
# ============================================================
# INITIALIZE TRACK MEMORY + REID GALLERY
# ============================================================

reid_gallery = {}
track_memory = {}
pending_candidates = []
persistent_rows = []


for track_id, box in enumerate(
    first_boxes
):

    embedding = extract_reid_embedding(
        frame_number=PERSISTENT_TEST_START,
        box=box
    )

    add_embedding_to_gallery(
        reid_gallery,
        track_id,
        embedding
    )

    track_memory[
        int(track_id)
    ] = {
        "last_frame":
            int(PERSISTENT_TEST_START),

        "last_box":
            np.asarray(
                box,
                dtype=np.float32
            ).copy(),

        "status":
            "active",

        "left_at_edge":
            is_near_frame_edge(
                box
            )
    }


print(
    "track_memory IDs:",
    sorted(
        track_memory.keys()
    )
)

print(
    "ReID gallery IDs:",
    sorted(
        reid_gallery.keys()
    )
)

print(
    "Next track ID:",
    next_track_id
)

NameError: name 'add_embedding_to_gallery' is not defined

In [1]:
# ============================================================
# PERSISTENT SAM2 + RF-DETR + LOST TRACK REID RECOVERY
# ============================================================

persistent_rows = []
pending_candidates = []

track_missed_refreshes = {
    int(track_id): 0
    for track_id in track_memory.keys()
}


NUM_TEST_FRAMES = (
    PERSISTENT_TEST_END
    - PERSISTENT_TEST_START
    + 1
)


segment_starts = range(
    0,
    NUM_TEST_FRAMES,
    REFRESH_INTERVAL
)


for segment_start_idx in tqdm(
    segment_starts,
    desc="Persistent SAM2 + ReID tracking"
):

    # ========================================================
    # SEGMENT LIMITS
    # ========================================================

    segment_end_idx = min(
        segment_start_idx
        + REFRESH_INTERVAL
        - 1,
        NUM_TEST_FRAMES - 1
    )

    real_segment_start = (
        PERSISTENT_TEST_START
        + segment_start_idx
    )

    real_segment_end = (
        PERSISTENT_TEST_START
        + segment_end_idx
    )


    matched_ids_segment = set()

    end_frame_matches = {}


    # ========================================================
    # SAM2 PROPAGATION
    # SAME PERSISTENT STATE
    # ========================================================

    max_frames_to_track = (
        segment_end_idx
        - segment_start_idx
    )


    for (
        local_frame_idx,
        object_ids,
        mask_logits
    ) in sam2_predictor.propagate_in_video(

        inference_state=
            persistent_state,

        start_frame_idx=
            segment_start_idx,

        max_frame_num_to_track=
            max_frames_to_track,

        reverse=False
    ):

        local_frame_idx = int(
            local_frame_idx
        )

        real_frame_number = (
            PERSISTENT_TEST_START
            + local_frame_idx
        )


        # ====================================================
        # RF-DETR DETECTIONS FOR CURRENT FRAME
        # ====================================================

        detections = (
            rfdetr_cache[
                real_frame_number
            ]
        )

        detector_boxes = (
            detections[
                "boxes"
            ]
        )

        detector_confidences = (
            detections[
                "confidences"
            ]
        )


        # ====================================================
        # SAM2 MASKS -> TEMPORARY BOXES
        # ====================================================

        sam2_boxes = []

        sam2_ids = []


        for object_index, object_id in enumerate(
            object_ids
        ):

            mask = (
                mask_logits[
                    object_index
                ] > 0.0
            )

            mask = (
                mask
                .detach()
                .cpu()
                .numpy()
            )

            box = mask_to_box(
                mask
            )

            if box is None:
                continue

            sam2_boxes.append(
                box
            )

            sam2_ids.append(
                int(object_id)
            )


        sam2_boxes = np.asarray(
            sam2_boxes,
            dtype=np.float32
        ).reshape(
            -1,
            4
        )


        used_detection_indices = set()


        # ====================================================
        # SAM2 -> RF-DETR ONE-TO-ONE MATCH
        # ====================================================

        if (
            len(sam2_boxes) > 0
            and
            len(detector_boxes) > 0
        ):

            matches = match_boxes(
                sam2_boxes,
                detector_boxes,
                iou_threshold=0.05,
                center_threshold=0.05
            )


            for (
                sam2_index,
                detection_index,
                match_iou,
                center_distance
            ) in matches:

                track_id = int(
                    sam2_ids[
                        sam2_index
                    ]
                )

                detector_box = np.asarray(
                    detector_boxes[
                        detection_index
                    ],
                    dtype=np.float32
                )

                confidence = float(
                    detector_confidences[
                        detection_index
                    ]
                )


                used_detection_indices.add(
                    detection_index
                )

                matched_ids_segment.add(
                    track_id
                )


                # =================================================
                # UPDATE TRACK MEMORY + REID GALLERY
                # =================================================

                update_track_memory(
                    track_id=
                        track_id,

                    frame_number=
                        real_frame_number,

                    detector_box=
                        detector_box
                )


                # =================================================
                # OUTPUT
                # ID = SAM2
                # BOX = RF-DETR
                # =================================================

                add_tracking_row(

                    frame_number=
                        real_frame_number,

                    track_id=
                        track_id,

                    box=
                        detector_box,

                    confidence=
                        confidence,

                    source=
                        "sam2+rfdetr",

                    sam2_iou=
                        float(match_iou),

                    center_distance=
                        float(center_distance)
                )


                # =================================================
                # SAVE MATCH AT REFRESH FRAME
                # =================================================

                if (
                    local_frame_idx
                    == segment_end_idx
                ):

                    end_frame_matches[
                        track_id
                    ] = (
                        detector_box.copy(),
                        confidence
                    )


        # ====================================================
        # RF-DETR DETECTIONS NOT MATCHED TO SAM2
        # ====================================================

        unmatched_detection_indices = [

            index

            for index in range(
                len(detector_boxes)
            )

            if index
            not in used_detection_indices
        ]


        # ====================================================
        # UPDATE CANDIDATE NEW / RETURNING PLAYERS
        # ====================================================

        update_pending_candidates(

            frame_number=
                real_frame_number,

            detector_boxes=
                detector_boxes,

            detector_confidences=
                detector_confidences,

            unmatched_detection_indices=
                unmatched_detection_indices
        )


    # ========================================================
    # REFRESH EXISTING SAM2 OBJECTS
    # USING RF-DETR BOXES
    # ========================================================

    for track_id, (
        detector_box,
        confidence
    ) in end_frame_matches.items():

        sam2_predictor.add_new_points_or_box(

            inference_state=
                persistent_state,

            frame_idx=
                segment_end_idx,

            obj_id=
                int(track_id),

            box=np.asarray(
                detector_box,
                dtype=np.float32
            )
        )


        track_missed_refreshes[
            track_id
        ] = 0


    # ========================================================
    # ACTIVE -> LOST
    # ========================================================

    for track_id in list(
        track_memory.keys()
    ):

        if (
            track_id
            not in end_frame_matches
        ):

            if (
                real_segment_end
                >
                track_memory[
                    track_id
                ][
                    "last_frame"
                ]
            ):

                track_memory[
                    track_id
                ][
                    "status"
                ] = "lost"


    # ========================================================
    # UPDATE MISSED REFRESH COUNTERS
    # ========================================================

    current_sam2_ids = [
        int(track_id)
        for track_id
        in persistent_state[
            "obj_ids"
        ]
    ]


    ids_to_remove = []


    for track_id in current_sam2_ids:

        if (
            track_id
            in end_frame_matches
        ):
            continue


        track_missed_refreshes[
            track_id
        ] = (
            track_missed_refreshes.get(
                track_id,
                0
            )
            + 1
        )


        if (
            track_missed_refreshes[
                track_id
            ]
            >= MAX_MISSED_REFRESHES
        ):

            ids_to_remove.append(
                track_id
            )


    # ========================================================
    # REMOVE STALE OBJECTS FROM ACTIVE SAM2 MEMORY
    # BUT KEEP REID MEMORY IN track_memory / gallery
    # ========================================================

    for track_id in ids_to_remove:

        try:

            sam2_predictor.remove_object(

                inference_state=
                    persistent_state,

                obj_id=
                    int(track_id),

                strict=False,

                need_output=False
            )

        except Exception as e:

            print(
                f"Could not remove ID "
                f"{track_id}: {e}"
            )


        track_missed_refreshes.pop(
            track_id,
            None
        )


        if (
            track_id
            in track_memory
        ):

            track_memory[
                track_id
            ][
                "status"
            ] = "lost"


    # ========================================================
    # CONFIRM NEW / RETURNING PLAYER CANDIDATES
    # ========================================================

    confirmed_candidates = []


    for candidate in pending_candidates:

        if (
            candidate[
                "hits"
            ]
            >= NEW_TRACK_CONFIRM_FRAMES

            and

            candidate[
                "last_frame"
            ]
            == real_segment_end
        ):

            confirmed_candidates.append(
                candidate
            )


    # ========================================================
    # LOST TRACK RECOVERY OR NEW ID
    # ========================================================

    for candidate in confirmed_candidates:

        (
            recovered_id,
            reid_similarity
        ) = find_lost_track_match(

            candidate,
            real_segment_end
        )


        # ====================================================
        # CASE 1 - RECOVER OLD ID
        # ====================================================

        if recovered_id is not None:

            track_id = int(
                recovered_id
            )

            print(
                f"RECOVERED ID "
                f"{track_id}"
                f" | frame "
                f"{real_segment_end}"
                f" | ReID "
                f"{reid_similarity:.3f}"
            )


        # ====================================================
        # CASE 2 - REALLY NEW PLAYER
        # ====================================================

        else:

            track_id = int(
                next_track_id
            )

            next_track_id += 1


            print(
                f"NEW ID "
                f"{track_id}"
                f" | frame "
                f"{real_segment_end}"
            )


        # ====================================================
        # ADD / RE-PROMPT OBJECT TO SAME SAM2 STATE
        # ====================================================

        sam2_predictor.add_new_points_or_box(

            inference_state=
                persistent_state,

            frame_idx=
                segment_end_idx,

            obj_id=
                track_id,

            box=np.asarray(
                candidate[
                    "box"
                ],
                dtype=np.float32
            )
        )


        # ====================================================
        # RESET MISSED COUNTER
        # ====================================================

        track_missed_refreshes[
            track_id
        ] = 0


        # ====================================================
        # UPDATE TRACK MEMORY
        # ====================================================

        track_memory[
            track_id
        ] = {

            "last_frame":
                real_segment_end,

            "last_box":
                np.asarray(
                    candidate[
                        "box"
                    ],
                    dtype=np.float32
                ).copy(),

            "status":
                "active",

            "left_at_edge":
                is_near_frame_edge(
                    candidate[
                        "box"
                    ]
                )
        }


        # ====================================================
        # UPDATE REID GALLERY
        # ====================================================

        candidate_embedding = (
            get_candidate_embedding(
                candidate
            )
        )


        add_embedding_to_gallery(

            reid_gallery,

            track_id,

            candidate_embedding
        )


        # ====================================================
        # BACKFILL CANDIDATE FRAMES
        # ====================================================

        for (
            history_frame,
            history_box,
            history_confidence
        ) in candidate[
            "history"
        ]:

            add_tracking_row(

                frame_number=
                    history_frame,

                track_id=
                    track_id,

                box=
                    history_box,

                confidence=
                    history_confidence,

                source=(
                    "reid_recovered"
                    if recovered_id
                    is not None
                    else
                    "new_confirmed"
                )
            )


        pending_candidates.remove(
            candidate
        )


    # ========================================================
    # PROGRESS LOG
    # ========================================================

    active_count = sum(
        1
        for memory
        in track_memory.values()
        if memory[
            "status"
        ] == "active"
    )

    lost_count = sum(
        1
        for memory
        in track_memory.values()
        if memory[
            "status"
        ] == "lost"
    )


    print(
        f"\nFrames "
        f"{real_segment_start}-"
        f"{real_segment_end}"
        f" | SAM2 objects: "
        f"{len(persistent_state['obj_ids'])}"
        f" | active: "
        f"{active_count}"
        f" | lost: "
        f"{lost_count}"
        f" | next ID: "
        f"{next_track_id}"
        f" | pending: "
        f"{len(pending_candidates)}"
    )

NameError: name 'track_memory' is not defined

In [ ]:
# ============================================================
# BUILD FINAL DATAFRAME
# ============================================================

persistent_df = pd.DataFrame(
    persistent_rows
)

persistent_df = (
    persistent_df
    .drop_duplicates(
        subset=[
            "frame",
            "track_id"
        ],
        keep="last"
    )
    .sort_values(
        [
            "frame",
            "track_id"
        ]
    )
    .reset_index(
        drop=True
    )
)

print(
    "Rows:",
    len(persistent_df)
)

print(
    "Unique IDs:",
    persistent_df[
        "track_id"
    ].nunique()
)

print(
    "Frame range:",
    persistent_df[
        "frame"
    ].min(),
    "->",
    persistent_df[
        "frame"
    ].max()
)